# Deterministic Fast-Path Routing with Fallback

When building LLM agents, you often encounter a common pattern: users frequently ask simple, deterministic questions (e.g., "What's the weather?" or "Check my calendar") alongside complex reasoning tasks. 

Routing every single query through a massive LLM just to pick a standard tool introduces unnecessary latency and API costs. However, completely replacing your agent with a deterministic router is dangerous, as you lose the LLM's ability to reason through ambiguous or multi-constraint requests.

This guide demonstrates **Deterministic Fast-Path Routing with Fallback** using `SynaptoRoute`. This pattern safely places a local semantic router *in front* of your LLM agent, providing three strict guarantees:

1. **High-Confidence Bypass:** Deterministic intents resolve locally in ~3ms, bypassing the LLM.
2. **Safe Abstention:** Low-margin, ambiguous, or multi-intent requests automatically fall back to the normal reasoning agent.
3. **First-Class Traceability:** Synthetic tool calls are explicitly tagged in LangSmith, so operators never mistakenly attribute a routed action to the LLM.


In [ ]:
%%capture --no-stderr
%pip install -U langgraph langchain-openai synaptoroute

## 1. Defining the Fast-Path Routes

We start by defining our deterministic tools. We configure `AdaptiveRouter` with explicit routes and a strict `margin`. The margin ensures that if a query overlaps multiple intents (or matches nothing strongly), the router safely abstains.


In [ ]:
import uuid
import os
from typing import Literal, Annotated, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from synaptoroute import AdaptiveRouter, Route

# Set up tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "SynaptoRoute-Fast-Path"

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# 1. Configure the semantic routes for deterministic tools
routes = [
    Route(
        name="get_weather",
        utterances=["What is the weather in Tokyo?", "tell me the weather", "is it raining today"],
        metadata={"tool_name": "get_weather"}
    ),
    Route(
        name="get_calendar",
        utterances=["What is on my calendar?", "Check my schedule", "Do I have any meetings"],
        metadata={"tool_name": "get_calendar"}
    )
]

# We enforce a strict 15% confidence margin between the top match and runner-up
router = AdaptiveRouter(margin=0.15)
for route in routes:
    router.add_route(route)


## 2. The Safe Interception Node

The interception node sits right after `START`. Its only job is to attempt a fast-path match. If it succeeds, it yields a synthetic `AIMessage` with a `tool_call`. If it fails or abstains, it yields nothing, safely passing the state to the LLM agent.

**Crucially, we inject router metrics into `response_metadata`.** If a tool call is produced by the semantic router, the LangSmith trace will explicitly show the route name, score, and margin. Otherwise, operators might attribute the action to the LLM and mistakenly tune the wrong layer.


In [ ]:
def synaptoroute_fast_path(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    
    # Only route Human messages
    if not isinstance(last_message, HumanMessage):
        return {"messages": []} 
        
    try:
        # Guarantee 1 & 2: Query the router. Returns None if confidence/margin is too low.
        route = router(last_message.content)
        
        if route is None:
            return {"messages": []} # Abstain: Let it flow to the LLM
            
        tool_name = route.metadata["tool_name"]
        
        # Guarantee 3: Traceability for LangSmith
        response_metadata = {
            "source": "SynaptoRoute",
            "route_name": route.name,
            "margin_configured": 0.15,
            "match_score": route.metadata.get("match_score"),
            "match_margin": route.metadata.get("match_margin"),
            "fast_path_triggered": True
        }
        
        synthetic_ai_msg = AIMessage(
            content="",
            tool_calls=[{
                "name": tool_name, 
                "args": {"query": last_message.content}, 
                "id": f"call_{uuid.uuid4().hex[:8]}", 
                "type": "tool_call"
            }],
            response_metadata=response_metadata
        )
        
        return {"messages": [synthetic_ai_msg]}
        
    except Exception:
        # In case of any unexpected failure, always fail open to the LLM
        return {"messages": []} 


## 3. Wiring the Graph

We use conditional edges to determine if the fast-path fired. If the state contains an `AIMessage` with tool calls, we skip the LLM and go straight to the tools!


In [ ]:
# Mock normal LLM and Tool nodes for demonstration
def normal_llm_agent(state: AgentState) -> dict:
    print("[NORMAL LLM] Processing query with full reasoning model.")
    return {"messages": [AIMessage(content="I reasoned through this.", tool_calls=[{"name": "get_weather", "args": {}, "id": "call_llm_1", "type": "tool_call"}])]}

def tools_node(state: AgentState) -> dict:
    print("[TOOLS] Tool node executed.")
    return {"messages": []}

def route_decision(state: AgentState) -> Literal["tools", "llm_agent"]:
    last_message = state["messages"][-1]
    # If the fast-path successfully generated a synthetic tool call, skip the LLM
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    return "llm_agent"

workflow = StateGraph(AgentState)
workflow.add_node("synaptoroute_node", synaptoroute_fast_path)
workflow.add_node("llm_agent", normal_llm_agent)
workflow.add_node("tools", tools_node)

workflow.add_edge(START, "synaptoroute_node")
workflow.add_conditional_edges("synaptoroute_node", route_decision)
workflow.add_edge("llm_agent", "tools")
workflow.add_edge("tools", END)

app = workflow.compile()


## 4. Testing the Guarantees

Fast paths are powerful exactly when abstention is first-class. Let's look at two queries to see how the graph handles them.

### Scenario A: Deterministic Match (Bypasses LLM)


In [ ]:
query = "What is the weather in Tokyo?"
result = app.invoke({"messages": [HumanMessage(content=query)]})

# Let's inspect the LangSmith traceability metadata attached to the synthetic message
last_ai_msg = [m for m in result["messages"] if isinstance(m, AIMessage)][0]
print(f"Traceability Metadata:\n{last_ai_msg.response_metadata}")


### Scenario B: Tricky Constraint (Router Abstains)
Here is a prompt that looks like weather/calendar, but contains an extra constraint requiring complex reasoning.


In [ ]:
query = "What is the weather like, and based on that and my calendar, should I wear a jacket to my meeting?"
result = app.invoke({"messages": [HumanMessage(content=query)]})

# Let's verify that the LLM took over
last_ai_msg = [m for m in result["messages"] if isinstance(m, AIMessage)][-1]
print(f"Fallback Successful. LLM Agent states:\n{last_ai_msg.content}")

# Outcome: 
# The router detects overlapping intents and low margin, so it ABSTAINS.
# It safely returns None, and the query is passed to the standard LLM agent 
# to properly reason through the constraints. Zero bugs, zero corrupted state.
